# Lesson 2 Lab: From Words to Numbers
### Bag of Words, TF-IDF, and Your First Spam Detector

Scam text messages trick people into losing money every day. In this lab, you will build a program that reads a text message and decides if it is **spam** or **ham** (a safe message).

**What you will do:**
1. Turn words into numbers with **Bag of Words**
2. Find important words with **TF-IDF**
3. Explore a real dataset of 5,572 text messages
4. Train and test a **spam detector**
5. Check the model's mistakes and try to fool it
6. Run experiments to make it better
7. Practice exercises

**How to use this notebook:** Run each cell from top to bottom with **Shift + Enter**. Read the comments (lines starting with `#`). They explain what each line does.

---
## Step 0: Setup

We will use these libraries:
- **scikit-learn** for turning text into numbers and for machine learning
- **pandas** for working with tables of data
- **matplotlib** for drawing charts

Google Colab already has all three. If you are using your own computer, run the cell below first.

In [ ]:
# Install the libraries (only needed on your own computer, not in Colab).
!pip install -q scikit-learn pandas matplotlib

In [ ]:
# Import the tools we need for this lab.
import pandas as pd                 # tables of data
import numpy as np                  # math with lists of numbers
import matplotlib.pyplot as plt     # charts

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("All libraries are ready!")

---
## Part 1: Why Do We Need Numbers?

A machine learning model is a **math formula**. You cannot multiply or add a word like "prize." So before a computer can judge a message, we must turn the words into numbers.

In [ ]:
# Try to do math with words. This will cause an error on purpose!
try:
    result = "free prize" * 0.5
except TypeError as error:
    print("Python says:", error)
    print("\nComputers cannot do math with words. We need numbers!")

---
## Part 2: Bag of Words

**Bag of Words** counts how many times each word appears in a message. It ignores the order of the words.

### 2.1 Bag of Words by hand (pure Python)
Let's build it ourselves first, so we understand exactly what happens.

In [ ]:
messages = ["win free money now",
            "free lunch at school now",
            "see you at school now"]

# STEP 1: Build the vocabulary (every unique word, sorted A to Z).
vocabulary = set()                     # a set keeps only unique items
for message in messages:
    for word in message.split():       # split the message into words
        vocabulary.add(word)
vocabulary = sorted(vocabulary)        # sort A to Z

print("Vocabulary:", vocabulary)
print("Number of words:", len(vocabulary))

In [ ]:
# STEP 2: Count each vocabulary word in each message.
for message in messages:
    words = message.split()
    # For each vocabulary word, count how many times it appears in this message.
    vector = [words.count(vocab_word) for vocab_word in vocabulary]
    print(f"{message:<26} -> {vector}")

# Each message is now a VECTOR: a list of numbers with the same length.

### 2.2 Bag of Words with scikit-learn
`CountVectorizer` does the same work in two lines. It also lowercases words and removes punctuation for us.

In [ ]:
vectorizer = CountVectorizer()

# fit      = learn the vocabulary
# transform = count the words in each message
X = vectorizer.fit_transform(messages)

print("Vocabulary:", vectorizer.get_feature_names_out())
print(X.toarray())    # .toarray() shows the full table of numbers

In [ ]:
# A pandas table makes the result easier to read.
bow_table = pd.DataFrame(X.toarray(),
                         columns=vectorizer.get_feature_names_out(),
                         index=messages)
bow_table

### 2.3 The weakness: word order is lost

In [ ]:
pair = ["dog bites man", "man bites dog"]
X_pair = CountVectorizer().fit_transform(pair)

print(X_pair.toarray())
print("\nSame numbers for both! Bag of Words cannot tell who bit whom.")

---
## Part 3: TF-IDF

**TF-IDF** gives each word an importance score.
- **TF (Term Frequency):** How often does the word appear in this message?
- **IDF (Inverse Document Frequency):** How rare is the word across all messages?

A word that appears in **every** message (like "now") gets a **low** score. A rare word (like "money") gets a **high** score.

In [ ]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(messages)

# idf_ holds the "rarity score" of each word.
idf_table = pd.DataFrame({"word": tfidf.get_feature_names_out(),
                          "IDF (rarity)": tfidf.idf_.round(2)})

# Sort from most common (low IDF) to most rare (high IDF).
idf_table.sort_values("IDF (rarity)")

In [ ]:
# Let's check the IDF formula ourselves for the word "now".
# scikit-learn uses: IDF = ln((1 + N) / (1 + df)) + 1
#   N  = total number of messages
#   df = number of messages that contain the word

N = 3          # we have 3 messages
df_now = 3     # "now" is in all 3 messages
df_money = 1   # "money" is in only 1 message

print("IDF of 'now':  ", round(np.log((1 + N) / (1 + df_now)) + 1, 2))
print("IDF of 'money':", round(np.log((1 + N) / (1 + df_money)) + 1, 2))

In [ ]:
# Now compare the final TF-IDF scores with Bag of Words.
tfidf_table = pd.DataFrame(X_tfidf.toarray().round(2),
                           columns=tfidf.get_feature_names_out(),
                           index=messages)
tfidf_table

# Look at the "now" column. It always has the lowest score in its row,
# because "now" appears in every message, so it is a weak clue.

---
## Part 4: Meet the Dataset

We will use the **SMS Spam Collection**, a free dataset of **5,572** real text messages. Each message has a label: `ham` (safe) or `spam`.

The messages come from the UK and Singapore, so you will see slang like "lor" and "wif."

In [ ]:
# The dataset is stored online as a tab-separated file (columns split by tabs).
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/.tsv"

try:
    # sep="\t" means the columns are separated by tabs.
    # header=None means the file has no title row, so we give the names ourselves.
    df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
    print("Dataset downloaded!")
except Exception as error:
    # Backup plan: if there is no internet, use a tiny practice dataset.
    print("Could not download the dataset, so we will use a small backup set.")
    print("Reason:", error)
    backup = [
        ("spam", "WINNER! You have been selected to receive a cash prize. Call now to claim"),
        ("spam", "URGENT! Your mobile number won a FREE trip. Text WIN to 80888"),
        ("spam", "Congratulations, claim your guaranteed reward today. Reply YES now"),
        ("spam", "FREE entry to win a new phone. Txt CLAIM to 81010 now"),
        ("spam", "You have won 1000 cash. To claim call our landline now"),
        ("spam", "Your account was selected for a prize. Click the link to claim"),
        ("ham", "Hey, are we still meeting at the library after class?"),
        ("ham", "Ok, I will call you later tonight"),
        ("ham", "Sorry I am late, the bus was slow"),
        ("ham", "Can you bring my notebook tomorrow?"),
        ("ham", "Mom says dinner is ready, come home soon"),
        ("ham", "Free lunch at school today, see you there"),
        ("ham", "I got home already, thanks for the ride"),
        ("ham", "Let's study for the quiz this weekend"),
        ("ham", "Happy birthday! Hope you have a great day"),
        ("ham", "What time does the game start later?"),
    ]
    df = pd.DataFrame(backup, columns=["label", "message"])

# .head() shows the first 5 rows of the table.
df.head()

In [ ]:
# How many messages do we have?
print("Total messages:", len(df))

# value_counts() counts how many of each label there are.
print(df["label"].value_counts())

# normalize=True turns the counts into percentages.
print((df["label"].value_counts(normalize=True) * 100).round(1))

In [ ]:
# Draw a bar chart of ham vs. spam.
counts = df["label"].value_counts()
plt.bar(counts.index, counts.values, color=["#13A89E", "#EF5B5B"])
plt.title("How many messages are ham and spam?")
plt.ylabel("Number of messages")
plt.show()

# Notice: there are MANY more ham messages than spam messages.
# Remember this. It will matter later!

In [ ]:
# Let's read some real spam messages.
spam_messages = df[df["label"] == "spam"]["message"]
for text in spam_messages.head(5):
    print("-", text)

In [ ]:
# Do spam messages tend to be longer? Let's count the characters.
df["length"] = df["message"].str.len()

# groupby() splits the table by label, then mean() finds the average.
print(df.groupby("label")["length"].mean().round(1))

---
## Part 5: Build the Spam Detector

### 5.1 Split the data: practice vs. exam
- **Training set (75%)**: the "practice questions." The model learns from these.
- **Test set (25%)**: the "final exam." The model has never seen these messages.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"],          # X = the inputs (the messages)
    df["label"],            # y = the answers (ham or spam)
    test_size=0.25,         # 25% goes to the test set
    random_state=42,        # makes the split the same every time we run it
    stratify=df["label"])   # keeps the same ham/spam mix in both sets

print("Training messages:", len(X_train))
print("Test messages:    ", len(X_test))

### 5.2 Turn the messages into numbers

In [ ]:
vectorizer = CountVectorizer()

# IMPORTANT:
# fit_transform on the TRAINING set: learn the vocabulary AND count words.
X_train_num = vectorizer.fit_transform(X_train)

# transform only on the TEST set: count words using the vocabulary we already learned.
# We never let the model "peek" at the exam while studying.
X_test_num = vectorizer.transform(X_test)

print("Training table size:", X_train_num.shape)   # (rows, columns)

# How many cells are zero? (This table is "sparse".)
total_cells = X_train_num.shape[0] * X_train_num.shape[1]
non_zero = X_train_num.nnz                           # nnz = number of non-zero cells
print(f"Percent of zeros: {100 * (1 - non_zero / total_cells):.1f}%")

### 5.3 Train the model
We use **Naive Bayes**. It learns how often each word appears in spam and in ham, then treats each word as a clue.

In [ ]:
# Create the model.
model = MultinomialNB()

# .fit() is where the LEARNING happens.
# The model studies the numbers (X_train_num) and the correct answers (y_train).
model.fit(X_train_num, y_train)

print("Training complete!")

### 5.4 Take the final exam

In [ ]:
# .predict() makes the model guess the label of each test message.
predictions = model.predict(X_test_num)

# Compare the guesses with the real answers.
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.1%}")

---
## Part 6: Is the Model Really Good?

### 6.1 The "lazy model" trap
What if a model just says **ham** every single time, without even reading the message?

In [ ]:
# Make a list that says "ham" for every test message.
lazy_predictions = ["ham"] * len(y_test)

lazy_accuracy = accuracy_score(y_test, lazy_predictions)
print(f"Lazy model accuracy: {lazy_accuracy:.1%}")
print(f"Our model accuracy:  {accuracy:.1%}")

# The lazy model catches ZERO spam, but its accuracy still looks high,
# because most messages are ham. Accuracy alone can fool us!

### 6.2 The confusion matrix
A confusion matrix shows exactly what kinds of mistakes the model made.

In [ ]:
# labels=["ham", "spam"] fixes the order of the rows and columns.
cm = confusion_matrix(y_test, predictions, labels=["ham", "spam"])
print(cm)

# Draw it as a colored chart.
ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"]).plot(cmap="Blues")
plt.title("Spam detector: where was it right or wrong?")
plt.show()

In [ ]:
# Pull the four numbers out of the matrix to explain them in words.
# .ravel() flattens the 2x2 table into a list of 4 numbers.
correct_ham, false_alarm, missed_spam, caught_spam = cm.ravel()

print(f"Safe messages kept correctly:   {correct_ham}")
print(f"False alarms (safe -> spam):    {false_alarm}")
print(f"Missed spam (spam -> inbox):    {missed_spam}")
print(f"Spam caught correctly:          {caught_spam}")
print(f"\nPercent of spam caught: {caught_spam / (caught_spam + missed_spam):.1%}")

> **Discuss:** Which mistake is worse for a spam filter: a **false alarm** or **missed spam**? Would your answer change for a bank or a hospital?

---
## Part 7: Test the Model with Your Own Messages

In [ ]:
# This helper function lets us test any message we want.
def check_message(text):
    numbers = vectorizer.transform([text])          # words -> numbers (note the [ ])
    label = model.predict(numbers)[0]               # the model's guess
    confidence = model.predict_proba(numbers).max() # how sure the model is (0 to 1)
    # Note: a very sure answer like 99.97% is rounded and shows as 100.0%.
    print(f"{label.upper():<5} ({confidence:.1%} sure)  <-  {text}")

check_message("Congratulations! You won a FREE phone. Call now to claim your prize!")
check_message("Hey, are we still meeting at the library after class?")
check_message("URGENT! Your account has been selected for a cash reward. Text WIN to 80888")
check_message("Free lunch at school today!")

In [ ]:
# YOUR TURN: write your own messages and test them!
check_message("Type your own message here")

### 7.1 Can we fool the model?

In [ ]:
check_message("Fr33 c4sh pr1ze!! cl1ck n0w")

# The model said HAM! It has never seen these strange spellings,
# so it found no spam clues. Real spammers use tricks like this.

---
## Part 8: Look Inside the Model

Which words does the model think are the strongest clues for spam and for ham?

In [ ]:
words = vectorizer.get_feature_names_out()

# feature_log_prob_ stores how likely each word is in each class.
# Row 0 = ham, row 1 = spam (alphabetical order of the labels).
# A big positive difference means "this word points to spam."
spam_score = model.feature_log_prob_[1] - model.feature_log_prob_[0]

# np.argsort() gives the positions from smallest to largest score.
top_spam = words[np.argsort(spam_score)[-15:]][::-1]   # the 15 largest, reversed
top_ham = words[np.argsort(spam_score)[:15]]           # the 15 smallest

print("Strongest SPAM clues:", list(top_spam))
print("\nStrongest HAM clues: ", list(top_ham))

---
## Part 9: Experiments

Machine learning engineers always test different options. Let's compare:
1. Bag of Words + Naive Bayes
2. TF-IDF + Naive Bayes
3. TF-IDF + Linear SVM (another model that learns a boundary line between spam and ham)

In [ ]:
# Each experiment is a pair: (a vectorizer, a model).
experiments = {
    "Bag of Words + Naive Bayes": (CountVectorizer(), MultinomialNB()),
    "TF-IDF + Naive Bayes":       (TfidfVectorizer(), MultinomialNB()),
    "TF-IDF + Linear SVM":        (TfidfVectorizer(), LinearSVC()),
}

results = []
for name, (vec, clf) in experiments.items():
    train_num = vec.fit_transform(X_train)    # learn vocabulary from practice set
    test_num = vec.transform(X_test)          # use it on the exam set
    clf.fit(train_num, y_train)               # train
    preds = clf.predict(test_num)             # predict
    cm = confusion_matrix(y_test, preds, labels=["ham", "spam"])
    results.append({"Experiment": name,
                    "Accuracy": f"{accuracy_score(y_test, preds):.1%}",
                    "False alarms": cm[0, 1],
                    "Spam missed": cm[1, 0]})

pd.DataFrame(results)

> **What did you notice?** TF-IDF sounds smarter, but with Naive Bayes it missed much more spam. A different model (Linear SVM) works well with TF-IDF. **Lesson:** never assume. Always test.

### 9.1 Bonus experiment: does cleaning the text help?

In Lesson 1, we learned lemmatization and stop word removal with spaCy. Let's see if cleaning the messages first improves our spam detector. This cell needs spaCy and may take about a minute to run.

In [ ]:
# Install and load spaCy (skip the install line if you already have it).
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import spacy

# We turn off the parts we do not need ("parser" and "ner") to make it faster.
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def clean(doc):
    # Keep the lowercase lemma of each token,
    # but skip stop words, punctuation, and extra spaces.
    return " ".join(token.lemma_.lower() for token in doc
                    if not token.is_stop and not token.is_punct and not token.is_space)

# nlp.pipe() processes many messages quickly in batches.
df["clean"] = [clean(doc) for doc in nlp.pipe(df["message"], batch_size=500)]

# Compare a few messages before and after cleaning.
df[["message", "clean"]].head()

In [ ]:
results = []
for column in ["message", "clean"]:
    # Use the same split settings, so the test is fair.
    Xtr, Xte, ytr, yte = train_test_split(df[column], df["label"], test_size=0.25,
                                          random_state=42, stratify=df["label"])
    vec = CountVectorizer()
    clf = MultinomialNB().fit(vec.fit_transform(Xtr), ytr)
    preds = clf.predict(vec.transform(Xte))
    cm = confusion_matrix(yte, preds, labels=["ham", "spam"])
    results.append({"Text used": column,
                    "Vocabulary size": len(vec.vocabulary_),
                    "Accuracy": f"{accuracy_score(yte, preds):.1%}",
                    "False alarms": cm[0, 1],
                    "Spam missed": cm[1, 0]})

pd.DataFrame(results)

# Cleaning makes the vocabulary smaller. Look closely at the two kinds of mistakes.
# Did cleaning help, hurt, or just trade one mistake for another?

---
## Part 10: Practice Exercises

Write your code where you see `# YOUR CODE HERE`. Try first, then check the answer key at the bottom.

In [ ]:
# Exercise 1: Build a Bag of Words table for these three messages.
#             Print the vocabulary and the table.
practice = ["buy cheap load now",
            "see you at the mall now",
            "cheap shoes at the mall now"]
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use TfidfVectorizer on the same messages.
#             Which word has the LOWEST IDF score? Why?
# YOUR CODE HERE

In [ ]:
# Exercise 3: Use check_message() to test 3 messages of your own:
#             one obvious spam, one normal message, and one tricky message.
# YOUR CODE HERE

In [ ]:
# Exercise 4 (Challenge): Train the "TF-IDF + Linear SVM" model.
#             Then write a function check_message_svm(text) that prints its prediction.
#             Hint: LinearSVC has .predict() but no .predict_proba().
# YOUR CODE HERE

---
## Answer Key

In [ ]:
# Exercise 1
practice = ["buy cheap load now", "see you at the mall now", "cheap shoes at the mall now"]
cv = CountVectorizer()
X_p = cv.fit_transform(practice)
print("Ex 1 vocabulary:", cv.get_feature_names_out())
print(X_p.toarray())

# Exercise 2
tv = TfidfVectorizer().fit(practice)
idf = pd.Series(tv.idf_.round(2), index=tv.get_feature_names_out()).sort_values()
print("\nEx 2 IDF scores:")
print(idf)
print("'now' has the lowest IDF because it appears in all 3 messages.")

# Exercise 3 (example answers)
print("\nEx 3:")
check_message("WIN a guaranteed cash prize! Call now to claim")
check_message("See you at practice later")
check_message("You won the class prize for best project!")

# Exercise 4
svm_vec = TfidfVectorizer()
svm_model = LinearSVC().fit(svm_vec.fit_transform(X_train), y_train)

def check_message_svm(text):
    label = svm_model.predict(svm_vec.transform([text]))[0]
    print(f"SVM says {label.upper():<5} <-  {text}")

print("\nEx 4:")
check_message_svm("Congratulations! You won a FREE phone. Call now to claim your prize!")
check_message_svm("Hey, are we still meeting at the library after class?")

---
## Great Job!

Today you:
- turned words into numbers with **Bag of Words** and **TF-IDF**,
- trained a real **spam detector** on 5,572 messages,
- learned that **accuracy alone can be misleading**,
- looked at the model's **mistakes** and tried to fool it,
- and ran **experiments** like a real machine learning engineer.

**Think about it:** How would you build a spam detector that works for Filipino or Taglish messages? What data would you need?